# Ultimate Attribute Opening Examples

This notebook demonstrates direct Ultimate Attribute Opening (UAO) workflows with `mmcfilters`.


In [ ]:
import numpy as np
import mmcfilters
import matplotlib
import math

import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (16,16)

from IPython import display
from os import listdir
import cv2 as cv


def load_grayscale(path):
    image = cv.imread(str(path), cv.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(path)
    return np.ascontiguousarray(image, dtype=np.uint8)


def create_component_tree(image, is_maxtree, radius=1.5):
    if is_maxtree:
        return mmcfilters.MorphologicalTreeFactory.createMaxTree(image, radius=radius)
    return mmcfilters.MorphologicalTreeFactory.createMinTree(image, radius=radius)


## Load the input image

The examples use a grayscale image so max-tree attributes can be interpreted directly against image intensity and component geometry.


In [ ]:
#input_image = load_grayscale("../dat/imgTeste.png")
input_image = load_grayscale("../dat/imgObjetos.png")
#input_image = load_grayscale("../dat/cable.png")

(num_rows, num_cols) = input_image.shape

#plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
#plt.title('input')

In [ ]:
tree = mmcfilters.MorphologicalTreeFactory.createMaxTree(input_image, radius=1.5)
attribute_indices, attribute_matrix = mmcfilters.Attribute.computeTopologyAttributes(tree, [mmcfilters.Attribute.BOX_HEIGHT])
node_attribute = attribute_matrix[:, attribute_indices['BOX_HEIGHT']]
attribute_filter = mmcfilters.AttributeFilters(tree)


In [ ]:
reconstructed_image = tree.reconstructionImage() 
filtered_image = attribute_filter.filteringSubtractiveRule(node_attribute > 500)

plt.subplot(1,3, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,3, 2)
plt.imshow(reconstructed_image.reshape(num_rows, num_cols), cmap='gray', vmax=255, vmin=0)
plt.title('rec')

plt.subplot(1,3, 3)
plt.imshow(filtered_image.reshape(num_rows, num_cols), cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter')


#np.sum(input_image.ravel() - reconstructed_image)

## Execute UAO directly

`UltimateAttributeOpening` computes the contrast and associated images directly for the selected criterion.


In [ ]:
mser_delta = 19
uao = mmcfilters.UltimateAttributeOpening(tree, node_attribute)
uao.executeWithMSER(num_rows, mser_delta)
roundtrip_contrast_image = uao.getMaxContrastImage()
roundtrip_associated_color_image = uao.getAssociatedColoredImage()

plt.subplot(1,2,1)
plt.imshow(roundtrip_contrast_image.reshape(num_rows, num_cols), cmap='gray', vmax=255, vmin=0)
plt.title('contrast')

plt.subplot(1,2,2)
plt.imshow(roundtrip_associated_color_image.reshape(num_rows, num_cols, 3), vmax=255, vmin=0)
plt.title('associated')

In [ ]:
mser_delta = 19
uao = mmcfilters.UltimateAttributeOpening(tree, node_attribute)

uao.executeWithMSER( num_rows, mser_delta )
associated_image = uao.getAssociatedColoredImage()
contrast_image = uao.getMaxContrastImage()

#import random
#colors = [ [random.randint(0, 255) for _ in range(3)] for _ in range(0, np.max(associated_image)+1)]
#rgb_contrast_image = [(0,0,0) if p == 0 else colors[p] for p in associated_image]

plt.subplot(1,2,1)
plt.imshow(contrast_image.reshape(num_rows, num_cols), cmap='gray', vmax=255, vmin=0)
plt.title('contrast')

plt.subplot(1,2,2)
plt.imshow(np.array(associated_image).reshape(num_rows, num_cols, 3), vmax=255, vmin=0)
plt.title('associated')

In [ ]:
np.sum(roundtrip_contrast_image - contrast_image)

## Repeat the workflow on a second image

The same steps are repeated on another input to check that the chosen parameters and attribute criterion generalize beyond the first example.


In [ ]:
input_image = load_grayscale("../dat/imgObjetos.png")
(num_rows, num_cols) = input_image.shape
tree = mmcfilters.MorphologicalTreeFactory.createMaxTree(input_image, radius=1.5)
attribute_indices, attribute_matrix = mmcfilters.Attribute.computeTopologyAttributes(tree, [mmcfilters.Attribute.BOX_HEIGHT])

uao = mmcfilters.UltimateAttributeOpening(tree, attribute_matrix[:, attribute_indices['BOX_HEIGHT']])
uao.execute(num_rows - 1)
contrast_image = uao.getMaxContrastImage()
associated_color_image = uao.getAssociatedColoredImage()

plt.subplot(1,2,1)
plt.imshow(contrast_image.reshape(num_rows, num_cols), cmap='gray', vmax=255, vmin=0)
plt.title('contrast')

plt.subplot(1,2,2)
plt.imshow(associated_color_image.reshape(num_rows, num_cols, 3), vmax=255, vmin=0)
plt.title('associated')

In [ ]:
associated_image = uao.getAssociatedImage().ravel()

import random
colors = [[random.randint(0, 255) for _ in range(3)] for _ in range(0, int(np.max(associated_image)) + 1)]
rgb_contrast_image = [(0, 0, 0) if p == 0 else colors[int(p)] for p in associated_image]

plt.imshow(np.array(rgb_contrast_image).reshape(num_rows, num_cols, 3), vmax=255, vmin=0)
plt.title('associated')

In [ ]:
maxCriterion = num_rows
mser_delta = 20

uao.executeWithMSER( int(maxCriterion), mser_delta )
contrast_image = uao.getMaxContrastImage()
associated_color_image = uao.getAssociatedColoredImage()

plt.subplot(1,2,1)
plt.imshow(contrast_image.reshape(num_rows, num_cols), cmap='gray', vmax=255, vmin=0)
plt.title('contrast')

plt.subplot(1,2,2)
plt.imshow(associated_color_image.reshape(num_rows, num_cols, 3), vmax=255, vmin=0)
plt.title('associated')

In [ ]:
np.min(associated_color_image), np.max(associated_color_image)

In [ ]:
uao = mmcfilters.UltimateAttributeOpening(tree, attribute_matrix[:,attribute_indices['BOX_HEIGHT']])
maxCriterion = num_rows
line = 1
for i in range(1, 6+1):
    mser_delta = int(i*3.8)
    uao.executeWithMSER( int(maxCriterion), mser_delta )
    associated_color_image = uao.getAssociatedColoredImage()
        
    plt.subplot(line,6, i)
    plt.imshow(associated_color_image.reshape(num_rows, num_cols, 3), vmax=255, vmin=0)
    plt.title('delta=' + str(mser_delta))